# Cortex AI Gateway + LangChain Agent + MCP

This notebook demonstrates how to build a **LangChain agent** that:

1. Routes inference through **Snowflake's Cortex AI Gateway** (using a Snowflake-hosted Claude model)
2. Connects to **Snowflake data via MCP** (Model Context Protocol) for tool use
3. Produces **observability traces** you can query in `SNOWFLAKE.TELEMETRY.AGENT_TRACE_TABLE`

## Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                     LangChain Agent (ReAct)                     │
│                                                                 │
│   LLM: ChatOpenAI ──► Cortex AI Gateway ──► Claude (Cortex)    │
│                                                                 │
│   Tools: MCP Client ──► Snowflake MCP Server ──► Cortex Agent  │
│          ├── Analyst: CUSTOMER_INTERACTIONS_SV (incl. churn)   │
│          └── Analyst: CONVERSATIONAL_BI_SV (sales KPIs)        │
│                                                                 │
│   ML: XGBoost ──► Model Registry ──► CHURN_PREDICTIONS          │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
            SNOWFLAKE.TELEMETRY.AGENT_TRACE_TABLE
            SNOWFLAKE.ACCOUNT_USAGE.AI_GATEWAY_USAGE_HISTORY
```

## Prerequisites

1. Run `setup.sql` in your Snowflake account to create the retail-athletic database and data, the two semantic views, the Cortex Agent, the MCP server, and the gateway configuration.
2. Create a [Personal Access Token (PAT)](https://docs.snowflake.com/en/user-guide/admin-pat) in Snowflake for API authentication.
3. Have your Snowflake account identifier (e.g., `myorg-myaccount`).

## 1. Install Dependencies

In [ ]:
%pip install langchain-openai langchain-mcp-adapters langgraph snowflake-connector-python snowflake-snowpark-python "snowflake-ml-python>=1.7.1" xgboost scikit-learn pandas

## 2. Configuration

The gateway and the MCP server both authenticate with a **programmatic access token (PAT)**.
Set two environment variables before starting Jupyter (or edit the defaults below):

```bash
export SNOWFLAKE_ACCOUNT=<org-account>          # e.g. myorg-myaccount
export SNOWFLAKE_USER=<user>
export SNOWFLAKE_PAT=<pat>                      # or SNOWFLAKE_PAT_FILE=/path/to/pat
```

Create a PAT with `ALTER USER <user> ADD PROGRAMMATIC ACCESS TOKEN ai_gateway_retail ROLE_RESTRICTION = 'SYSADMIN' DAYS_TO_EXPIRY = 7;`
(PATs are subject to your account's network and authentication policies.)


In [1]:
import os
from pathlib import Path

SNOWFLAKE_ACCOUNT = os.environ["SNOWFLAKE_ACCOUNT"]
SNOWFLAKE_USER = os.environ["SNOWFLAKE_USER"]
SNOWFLAKE_PAT = os.environ.get("SNOWFLAKE_PAT") or Path(
    os.path.expanduser(os.environ.get("SNOWFLAKE_PAT_FILE", "~/.snowflake/ai_gateway_retail.pat"))
).read_text().strip()

# Hostnames need hyphens, not underscores, for a valid TLS certificate.
# Use the endpoint from SHOW AI GATEWAYS if your account uses private connectivity.
SF_HOST = f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com".replace("_", "-").lower()
INFERENCE_BASE_URL = f"https://{SF_HOST}/api/v2/aigateways/SNOWFLAKE/v1"

MODEL = "openai-gpt-5.4"  # any model the gateway spec allows ('*' = all account models)

print(f"Account:       {SNOWFLAKE_ACCOUNT}")
print(f"User:          {SNOWFLAKE_USER}")
print(f"Inference URL: {INFERENCE_BASE_URL}")
print(f"Model:         {MODEL}")


Account:       SFSENORTHAMERICA-JGARCIA_DEMO_USWEST2
User:          jgarcia
Inference URL: https://sfsenorthamerica-jgarcia-demo-uswest2.snowflakecomputing.com/api/v2/aigateways/SNOWFLAKE/v1
Model:         openai-gpt-5.4


## 3. Initialize the Gateway LLM

The Cortex AI Gateway exposes an **OpenAI-compatible** API. We use `ChatOpenAI` from LangChain,
pointing `base_url` at the inference endpoint and passing the PAT as the API key.

The `model` parameter is the actual model name (e.g., `claude-sonnet-4-6`). The gateway's
`models` allowlist controls which models are permitted — our spec uses `'*'` to allow all.

### Trace context propagation

For the gateway to group all inference calls from one agent turn under a single
`trace_id`, the client must send a [W3C `traceparent`](https://www.w3.org/TR/trace-context/) header on every request. Without it, each call gets its own trace ID and
they appear as unrelated events in `AGENT_TRACE_TABLE`.

We define a `make_llm()` factory that accepts an optional `traceparent` header.
In `run_agent()` (Section 5) we generate a fresh traceparent per invocation so
all LLM calls within one turn share the same trace.

In [2]:
import uuid
from langchain_openai import ChatOpenAI


def make_traceparent() -> tuple[str, str]:
    """Generate a W3C traceparent header value and return (header_value, trace_id).

    Format: {version}-{trace_id}-{parent_id}-{flags}
    See https://www.w3.org/TR/trace-context/#traceparent-header
    """
    trace_id = uuid.uuid4().hex  # 32 hex chars
    parent_id = uuid.uuid4().hex[:16]  # 16 hex chars
    header = f"00-{trace_id}-{parent_id}-01"
    return header, trace_id


def make_llm(traceparent: str | None = None) -> ChatOpenAI:
    """Create a ChatOpenAI pointed at the Cortex AI Gateway.

    When *traceparent* is provided it is sent as a default header on every
    request so the gateway groups all inference calls under one trace_id.
    """
    headers = {}
    if traceparent:
        headers["traceparent"] = traceparent
    return ChatOpenAI(
        model=MODEL,
        base_url=INFERENCE_BASE_URL,
        api_key=SNOWFLAKE_PAT,
        temperature=0,
        max_completion_tokens=4096,  # GPT-5 family rejects max_tokens
        default_headers=headers or None,
    )


# Quick test - this goes through the gateway
llm = make_llm()
response = llm.invoke("What is sell-through rate in retail? Answer in one sentence.")
print(response.content)

Sell-through rate in retail is the percentage of inventory received that is sold to customers during a specific period, usually calculated as units sold divided by units received, multiplied by 100.


## 4. Connect MCP Tools

We use `langchain-mcp-adapters` to connect to the **Snowflake MCP server** object we created
in `setup.sql` (`RETAIL_MCP`). This MCP server exposes one tool:

- **`retail_analytics_agent`** (Cortex Agent `RETAIL_ANALYTICS_AGENT`) — orchestrates two Cortex Analyst tools, each of which generates and runs SQL against a semantic view: `CUSTOMER_INTERACTIONS_SV` (customers, engagement, campaigns, model churn scores) and `CONVERSATIONAL_BI_SV` (revenue, orders, margin, stores, product lines)

There is deliberately no raw SQL tool: exposed directly, Analyst only returns SQL text, and a
`SYSTEM_EXECUTE_SQL` tool would let the client run arbitrary SQL around the semantic views.

The MCP server endpoint follows the pattern:
```
https://<account>.snowflakecomputing.com/api/v2/databases/<db>/schemas/<schema>/mcp-servers/<name>/sse
```

`langchain-mcp-adapters` supports SSE transport directly — no local bridge or npx needed.

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# MCP server endpoint — points to our RETAIL_MCP server object in Snowflake
# Snowflake hostnames require hyphens (not underscores) for valid SSL certs
MCP_HOST = f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com".replace('_', '-')
MCP_ENDPOINT = (
    f"https://{MCP_HOST}"
    f"/api/v2/databases/CORTEX_GATEWAY_RETAIL_LAB/schemas/PUBLIC/mcp-servers/RETAIL_MCP"
)

# Streamable HTTP transport — connects directly to the Snowflake MCP server
mcp_config = {
    "snowflake": {
        "transport": "http",
        "url": MCP_ENDPOINT,
        "headers": {"Authorization": f"Bearer {SNOWFLAKE_PAT}"},
    }
}

print("MCP server configured (will connect when agent starts)")
print(f"  Transport: http")
print(f"  Endpoint:  {MCP_ENDPOINT}")

MCP server configured (will connect when agent starts)
  Transport: http
  Endpoint:  https://SFSENORTHAMERICA-JGARCIA-DEMO-USWEST2.snowflakecomputing.com/api/v2/databases/CORTEX_GATEWAY_RETAIL_LAB/schemas/PUBLIC/mcp-servers/RETAIL_MCP


## 5. Build the LangChain Agent

We create a **ReAct agent** using LangGraph. The agent:
1. Receives a user question
2. Calls the `retail_analytics_agent` MCP tool; the Cortex Agent behind it picks the customer analyst, the business KPI analyst, or both
3. Routes LLM inference through the Cortex AI Gateway
4. Synthesizes a final answer from the tool results

In [7]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """You are a retail analytics assistant for an athletic apparel brand. For any
question about sales, products, stores, customers, campaigns or churn risk, call
retail_analytics_agent with the user's question; it queries governed Snowflake data and returns
the answer.

Always cite specific numbers when available. Round currency to 2 decimal places.
The data covers October 2024 through September 2025."""

# Initialize the MCP client and load tools once
mcp_client = MultiServerMCPClient(mcp_config)
tools = await mcp_client.get_tools()
print(f"Loaded {len(tools)} MCP tools: {[t.name for t in tools]}")


async def run_agent(question: str):
    """Run the agent against a question.

    Each invocation creates a fresh LLM with a unique W3C traceparent header
    so that every inference call the agent makes during this turn is grouped
    under one trace_id in AGENT_TRACE_TABLE.
    """
    traceparent, trace_id = make_traceparent()
    traced_llm = make_llm(traceparent=traceparent)
    print(f"Trace ID: {trace_id}")

    agent = create_react_agent(
        model=traced_llm,
        tools=tools,
        prompt=SYSTEM_PROMPT,
    )

    result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})

    # Print tool calls and responses for visibility
    for msg in result["messages"]:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"Tool call: {tc['name']}({tc['args']})")
        if msg.type == 'tool':
            print(f"Tool response ({msg.name}): {msg.content[:500]}")
            print()

    final_message = result["messages"][-1]
    print("─" * 60)
    print(f"Q: {question}")
    print("─" * 60)
    print(final_message.content)
    return result

Loaded 1 MCP tools: ['retail_analytics_agent']


## 6. Run Queries

Let's ask the agent some retail analytics questions. Each request flows through:

**User → LangChain Agent → AI Gateway (Claude) → MCP Tools (Snowflake) → Response**

All inference calls are logged by the gateway for observability.

In [10]:
# Query 1: Executive KPI question (the agent uses business_kpi_analyst)
result1 = await run_agent("What was net revenue and gross margin by channel during the 2024 holiday season (November-December)?")

Trace ID: 3ce4461136ef44bebd555d52138c504f


/var/folders/k_/513c7sz949b5mxtzyn8z2sjm0000gn/T/ipykernel_22223/547535887.py:28: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Tool call: retail_analytics_agent({'text': 'What was net revenue and gross margin by channel during the 2024 holiday season (November-December)?'})
Tool response (retail_analytics_agent): [{'type': 'text', 'text': '{"content":[{"thinking":{"text":"\\nThe user wants net revenue and gross margin by channel for Nov-Dec 2024. Use business_kpi_analyst tool.\\n"},"type":"thinking"},{"tool_use":{"client_side_execute":false,"input":{"pruning_question":"What was net revenue and gross margin by channel during November and December 2024?"},"name":"business_kpi_analyst","tool_use_id":"toolu_bdrk_01GvCsbGpXrakYbwkfqfKv4N","type":"system_agentic_semantic_context"},"type":"tool_use"},{"tool_result":{"content":[{"json":{"custom_instructions":{"question_categorization":"This view covers retail business analytics: revenue, orders, margins, marketing efficiency, and store performance. For customer churn, lifetime value, or ML predictions, direct users to the Customer Interactions semantic view.","sql_gen

In [11]:
# Query 2: Customer question (the agent uses customer_interactions_analyst)
result2 = await run_agent("Which customer segments have the highest churn risk, and how many customers in each are predicted to churn?")

Trace ID: 2d60cfed924d4a81b763a481dc37d5aa


/var/folders/k_/513c7sz949b5mxtzyn8z2sjm0000gn/T/ipykernel_22223/547535887.py:28: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Tool call: retail_analytics_agent({'text': 'Which customer segments have the highest churn risk, and how many customers in each are predicted to churn?'})
Tool response (retail_analytics_agent): [{'type': 'text', 'text': '{"content":[{"thinking":{"text":"\\nUse customer_interactions_analyst for churn risk by segment using model churn score (avg_model_churn_score, predicted_churners).\\n"},"type":"thinking"},{"tool_use":{"client_side_execute":false,"input":{"pruning_question":"Which customer segments have the highest average model churn score and predicted churners count?"},"name":"customer_interactions_analyst","tool_use_id":"toolu_bdrk_015NX5riDqjxweJg8wSviQfD","type":"system_agentic_semantic_context"},"type":"tool_use"},{"tool_result":{"content":[{"json":{"pruned_note":"**Note**: This semantic model fits within the agent\'s token budget, so it is shown in full. You do not need to set pruning_question on future calls referencing this same semantic model.","relationships":[{"join_type"

In [12]:
# Query 3: Cross-domain question (the agent uses both analysts)
result3 = await run_agent(
    "Which product line has the highest net revenue, and what is its average customer rating? "
    "Also give our marketing ROAS by campaign type."
)

Trace ID: 4b80f5237d374ec9b47106228842d10d


/var/folders/k_/513c7sz949b5mxtzyn8z2sjm0000gn/T/ipykernel_22223/547535887.py:28: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Tool call: retail_analytics_agent({'text': 'Which product line has the highest net revenue, and what is its average customer rating? Also give our marketing ROAS by campaign type.'})
Tool response (retail_analytics_agent): [{'type': 'text', 'text': '{"content":[{"text":"I\'ll pull net revenue by product line (which includes ratings) and marketing ROAS by campaign type.","type":"text"},{"tool_use":{"client_side_execute":false,"input":{"pruning_question":"What is marketing ROAS by campaign type?"},"name":"customer_interactions_analyst","tool_use_id":"toolu_bdrk_0119F1CaX7HXJYndzcfJAD3j","type":"system_agentic_semantic_context"},"type":"tool_use"},{"tool_use":{"client_side_execute":false,"input":{"pruning_question":"Which product line has the highest net revenue and average customer rating?"},"name":"business_kpi_analyst","tool_use_id":"toolu_bdrk_01GEK7EsVvM83ihKAM8xz8DC","type":"system_agentic_semantic_context"},"type":"tool_use"},{"tool_result":{"content":[{"json":{"custom_instructions

## 7. Churn model: train, register, score, monitor

Query 2 answered from `CHURN_PREDICTIONS`, which `setup.sql` seeds from the legacy rule-based score (`MODEL_VERSION = 'LEGACY_SEED'`). Here we replace it with a real model, entirely in Snowflake:

1. Build a feature view in SQL (`CHURN_FEATURES_V`)
2. Train an XGBoost classifier and compare it to the legacy score
3. Log it to the **Snowflake Model Registry** (`CHURN_PREDICTION_MODEL` / `V1`)
4. Run **batch inference in the warehouse** with `mv.run()` → `CHURN_PREDICTIONS` (`MODEL_VERSION = 'V1'`)
5. Attach a **Model Monitor** (performance and drift against a training baseline)

The semantic view reads `CHURN_PREDICTIONS`, so the agent behind the MCP server picks up the new scores with no change to the agent, the MCP server, or the client. Re-running this section replaces the model and the monitor.

In [13]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F

DATABASE, SCHEMA, WAREHOUSE = "CORTEX_GATEWAY_RETAIL_LAB", "PUBLIC", "GATEWAY_RETAIL_WH"
MODEL_NAME, VERSION_NAME, MONITOR_NAME = "CHURN_PREDICTION_MODEL", "V1", "CHURN_MODEL_MONITOR"

session = Session.builder.configs({
    "account": SNOWFLAKE_ACCOUNT, "user": SNOWFLAKE_USER,
    "authenticator": "PROGRAMMATIC_ACCESS_TOKEN", "token": SNOWFLAKE_PAT,
    "warehouse": WAREHOUSE, "database": DATABASE, "schema": SCHEMA,
}).create()
print(session.get_current_role(), session.get_fully_qualified_current_schema())

# Features are computed in SQL so training and in-warehouse scoring share one definition.
# Label: CHURNED = CHURN_RISK_SCORE >= 0.5. LEGACY_CHURN_SCORE is kept only for comparison.
session.sql(f"""
CREATE OR REPLACE VIEW {DATABASE}.{SCHEMA}.CHURN_FEATURES_V AS
SELECT
    c.CUSTOMER_ID,
    m.DAYS_SINCE_LAST_PURCHASE::FLOAT                             AS DAYS_SINCE_LAST_PURCHASE,
    m.PURCHASE_FREQUENCY::FLOAT                                   AS PURCHASE_FREQUENCY,
    m.AVG_ORDER_VALUE::FLOAT                                      AS AVG_ORDER_VALUE,
    m.TOTAL_SPEND_12M::FLOAT                                      AS TOTAL_SPEND_12M,
    m.PRODUCT_DIVERSITY_SCORE::FLOAT                              AS PRODUCT_DIVERSITY_SCORE,
    m.CHANNEL_DIVERSITY_SCORE::FLOAT                              AS CHANNEL_DIVERSITY_SCORE,
    m.RETURN_RATE::FLOAT                                          AS RETURN_RATE,
    m.EMAIL_ENGAGEMENT_RATE::FLOAT                                AS EMAIL_ENGAGEMENT_RATE,
    m.WEB_VISITS_30D::FLOAT                                       AS WEB_VISITS_30D,
    c.NPS_SCORE::FLOAT                                            AS NPS_SCORE,
    c.TOTAL_ORDERS::FLOAT                                         AS TOTAL_ORDERS,
    c.LIFETIME_VALUE::FLOAT                                       AS LIFETIME_VALUE,
    DATEDIFF('day', c.FIRST_PURCHASE_DATE, m.FEATURE_DATE)::FLOAT AS TENURE_DAYS,
    IFF(c.OPT_IN_EMAIL, 1, 0)::FLOAT                              AS OPT_IN_EMAIL,
    IFF(c.OPT_IN_SMS, 1, 0)::FLOAT                                AS OPT_IN_SMS,
    IFF(c.PREFERRED_CHANNEL = 'Retail Store', 1, 0)::FLOAT        AS PREFERS_RETAIL,
    IFF(c.REGION = 'International', 1, 0)::FLOAT                  AS IS_INTERNATIONAL,
    IFF(c.CHURN_RISK_SCORE >= 0.5, 1, 0)                          AS CHURNED,
    m.CHURN_PROBABILITY::FLOAT                                    AS LEGACY_CHURN_SCORE
FROM {DATABASE}.{SCHEMA}.DIM_CUSTOMER c
JOIN {DATABASE}.{SCHEMA}.ML_CUSTOMER_FEATURES m ON m.CUSTOMER_ID = c.CUSTOMER_ID
""").collect()

df = session.table("CHURN_FEATURES_V").to_pandas()
FEATURES = [c for c in df.columns if c not in ("CUSTOMER_ID", "CHURNED", "LEGACY_CHURN_SCORE")]
print(df.shape, "| churn rate:", round(df.CHURNED.mean(), 3))

DatabaseError: 394400 (08001): Failed to connect to DB: SFSENORTHAMERICA-JGARCIA_DEMO_USWEST2.snowflakecomputing.com:443. Programmatic access token is invalid. [2f8614a7-938a-4406-9750-980b3ef8f507]

In [ ]:
# Train XGBoost and compare it to the legacy rule-based score
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df.CHURNED)
X_train, y_train = train_df[FEATURES], train_df.CHURNED
X_test,  y_test  = test_df[FEATURES],  test_df.CHURNED

model = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42)
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
auc, ap = roc_auc_score(y_test, proba), average_precision_score(y_test, proba)
legacy_auc = roc_auc_score(y_test, test_df.LEGACY_CHURN_SCORE)
print(f"XGBoost ROC AUC: {auc:.3f}   (legacy score: {legacy_auc:.3f})   PR AUC: {ap:.3f}")
pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False).head(8)

In [ ]:
# Log to the Model Registry, then batch-score every customer inside the warehouse
from snowflake.ml.registry import Registry
from snowflake.ml.model import type_hints

reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)
session.sql(f"DROP MODEL MONITOR IF EXISTS {MONITOR_NAME}").collect()
models = reg.show_models()  # empty frame with no columns when the schema has no models
if not models.empty and MODEL_NAME in models["name"].str.upper().values:
    reg.delete_model(MODEL_NAME)

mv = reg.log_model(
    model, model_name=MODEL_NAME, version_name=VERSION_NAME,
    sample_input_data=X_train.head(100), target_platforms=["WAREHOUSE"],
    task=type_hints.Task.TABULAR_BINARY_CLASSIFICATION,
    comment="XGBoost churn classifier on behavioural + engagement features",
    metrics={"roc_auc": round(auc, 4), "pr_auc": round(ap, 4), "legacy_roc_auc": round(legacy_auc, 4)},
)

scored = mv.run(session.table("CHURN_FEATURES_V"), function_name="predict_proba")
prob_col = [c for c in scored.columns if c.strip('"').upper().endswith("OUTPUT_FEATURE_1")][0]
predictions = (scored
    .with_column("CHURN_SCORE", F.round(F.col(prob_col), 4))
    .with_column("CHURN_PREDICTION", F.iff(F.col("CHURN_SCORE") >= 0.5, 1, 0))
    .with_column("MODEL_VERSION", F.lit(VERSION_NAME))
    .with_column("SCORED_AT", F.current_timestamp().cast("TIMESTAMP_NTZ")))

# Same columns as the seed table, so the semantic view keeps working unchanged
predictions.select("CUSTOMER_ID", "CHURN_SCORE", "CHURN_PREDICTION", "MODEL_VERSION", "SCORED_AT") \
           .write.mode("overwrite").save_as_table("CHURN_PREDICTIONS")

session.sql("""SELECT MODEL_VERSION, CHURN_PREDICTION, COUNT(*) AS customers, ROUND(AVG(CHURN_SCORE), 3) AS avg_score
               FROM CHURN_PREDICTIONS GROUP BY 1, 2 ORDER BY 2""").to_pandas()

In [ ]:
# Model Monitor: baseline = training rows, source = held-out rows spread over 4 weeks
# mv.run() keeps the input columns, so predictions already carry features and CHURNED
model_cols = FEATURES + ["CHURN_SCORE", "CHURN_PREDICTION", "CHURNED"]

def monitor_frame(ids):
    return (predictions.filter(F.col("CUSTOMER_ID").isin(list(ids)))
            .select("CUSTOMER_ID", *model_cols,
                    F.dateadd("day", -(F.abs(F.hash(F.col("CUSTOMER_ID"))) % 28),
                              F.to_timestamp_ntz(F.lit("2025-09-30"))).alias("SCORED_AT")))

monitor_frame(train_df.CUSTOMER_ID).write.mode("overwrite").save_as_table("CHURN_MONITOR_BASELINE")
monitor_frame(test_df.CUSTOMER_ID).write.mode("overwrite").save_as_table("CHURN_MONITOR_SOURCE")

session.sql(f"""
CREATE OR REPLACE MODEL MONITOR {MONITOR_NAME} WITH
    MODEL = {MODEL_NAME} VERSION = '{VERSION_NAME}' FUNCTION = 'predict'
    SOURCE = {DATABASE}.{SCHEMA}.CHURN_MONITOR_SOURCE
    BASELINE = {DATABASE}.{SCHEMA}.CHURN_MONITOR_BASELINE
    WAREHOUSE = {WAREHOUSE} REFRESH_INTERVAL = '1 day' AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = SCORED_AT ID_COLUMNS = ('CUSTOMER_ID')
    PREDICTION_SCORE_COLUMNS = ('CHURN_SCORE') PREDICTION_CLASS_COLUMNS = ('CHURN_PREDICTION')
    ACTUAL_CLASS_COLUMNS = ('CHURNED')
""").collect()
mon = session.sql(f"SHOW MODEL MONITORS LIKE '{MONITOR_NAME}'").to_pandas()
mon.columns = [c.strip('"').lower() for c in mon.columns]
mon[["name", "monitor_state"]]

### Ask again through the gateway

The same question as Query 2, unchanged client. The agent now answers from the registered model's scores (`MODEL_VERSION = 'V1'`). In Snowsight, **AI & ML → Models → CHURN_PREDICTION_MODEL → Monitors** shows the drift and performance dashboards.

In [ ]:
result4 = await run_agent("Which customer segments have the highest churn risk, and how many customers in each are predicted to churn?")

## 8. Observability — AI Gateway Traces

Every request through the AI Gateway is recorded with OpenTelemetry traces. These land in
the `AGENT_TRACE_TABLE('SNOWFLAKE')` table function, where each row is a **span** in a trace.

Key columns:
- `TRACE_ID` — groups all spans from one end-to-end request
- `SPAN_ID` — unique identifier for each operation
- `RECORD_ATTRIBUTES` — JSON with model name, token counts, etc.
- `TIMESTAMP` — when the span was recorded

Because we enabled `capture_payload.request_response: true` in the gateway spec,
request/response content is also captured in the trace attributes.

Let's query the observability data for the requests we just made.

In [ ]:
import snowflake.connector
import json

conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT,
    user=SNOWFLAKE_USER,
    authenticator="PROGRAMMATIC_ACCESS_TOKEN",
    token=SNOWFLAKE_PAT,
    warehouse="GATEWAY_RETAIL_WH",
)
cur = conn.cursor()
print(f"Connected as {SNOWFLAKE_USER} via PAT.")

In [ ]:
# Recent AI Gateway traces
cur.execute("""
    SELECT
        TRACE:trace_id::STRING AS TRACE_ID,
        RECORD:name::STRING AS SPAN_NAME,
        RECORD_ATTRIBUTES:"gen_ai.request.model"::STRING AS MODEL,
        RECORD_ATTRIBUTES:"gen_ai.usage.input_tokens"::INT AS IN_TOK,
        RECORD_ATTRIBUTES:"gen_ai.usage.output_tokens"::INT AS OUT_TOK,
        RECORD_ATTRIBUTES:"http.status_code"::INT AS HTTP
    FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
    WHERE TIMESTAMP > DATEADD('hour', -1, CURRENT_TIMESTAMP())
    ORDER BY TIMESTAMP DESC
    LIMIT 15
""")

rows = cur.fetchall()
print(f"Found {len(rows)} recent gateway trace spans\n")
print(f"{'TRACE_ID':>36} | {'SPAN_NAME':>25} | {'MODEL':>16} | {'IN':>6} | {'OUT':>6} | {'HTTP':>4}")
print("─" * 100)
for r in rows:
    print(f"{str(r[0]):>36} | {str(r[1]):>25} | {str(r[2]):>16} | {str(r[3]):>6} | {str(r[4]):>6} | {str(r[5]):>4}")

In [ ]:
# Credit usage summary from AI_GATEWAY_USAGE_HISTORY
# OPERATION_DETAILS is a JSON object keyed by model name with token counts
cur.execute("""
    SELECT
        f.key AS MODEL,
        COUNT(*) AS REQUEST_COUNT,
        SUM(f.value:"input_tokens"::INT) AS TOTAL_INPUT_TOKENS,
        SUM(f.value:"output_tokens"::INT) AS TOTAL_OUTPUT_TOKENS,
        SUM(CREDITS) AS TOTAL_CREDITS
    FROM SNOWFLAKE.ACCOUNT_USAGE.AI_GATEWAY_USAGE_HISTORY,
        LATERAL FLATTEN(input => OPERATION_DETAILS) f
    WHERE START_TIME > DATEADD('day', -1, CURRENT_TIMESTAMP())
    GROUP BY f.key
    ORDER BY TOTAL_CREDITS DESC
""")

rows = cur.fetchall()
print("AI Gateway Usage Summary (last 24h)\n")
print(f"{'MODEL':>20} | {'REQUESTS':>8} | {'IN_TOKENS':>10} | {'OUT_TOKENS':>10} | {'CREDITS':>10}")
print("─" * 70)
for row in rows:
    print(f"{str(row[0]):>20} | {row[1]:>8} | {row[2]:>10} | {row[3]:>10} | {row[4]:>10.6f}")

In [ ]:
# Detailed trace view — follow a single request end-to-end
cur.execute("""
    SELECT TRACE:trace_id::STRING AS TRACE_ID
    FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
    WHERE TIMESTAMP > DATEADD('hour', -1, CURRENT_TIMESTAMP())
    ORDER BY TIMESTAMP DESC
    LIMIT 1
""")
latest_trace = cur.fetchone()

if latest_trace:
    trace_id = latest_trace[0]
    print(f"Trace detail for: {trace_id}\n")

    cur.execute(f"""
        SELECT
            TRACE:span_id::STRING AS SPAN_ID,
            RECORD:name::STRING AS SPAN_NAME,
            TIMESTAMP,
            RECORD_ATTRIBUTES
        FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
        WHERE TRACE:trace_id::STRING = '{trace_id}'
        ORDER BY TIMESTAMP
    """)

    spans = cur.fetchall()
    for span in spans:
        attrs = json.loads(span[3]) if span[3] else {}
        print(f"  Span: {span[1]}")
        print(f"    Timestamp: {span[2]}")
        if "gen_ai.usage.input_tokens" in attrs:
            print(f"    Tokens: {attrs['gen_ai.usage.input_tokens']} in / {attrs['gen_ai.usage.output_tokens']} out")
        if "gen_ai.request.model" in attrs:
            print(f"    Model: {attrs['gen_ai.request.model']}")
        print()
else:
    print("No traces found yet — data may take a few minutes to appear.")

In [ ]:
# Deep dive: reconstruct the full agent conversation chain from a single trace
# This shows every reasoning step — tool calls, tool responses, and final answer.
cur.execute("""
    SELECT TRACE:trace_id::STRING AS TRACE_ID
    FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
    WHERE TIMESTAMP > DATEADD('hour', -2, CURRENT_TIMESTAMP())
      AND RECORD_ATTRIBUTES:"gen_ai.usage.input_tokens"::INT > 2000
    ORDER BY TIMESTAMP DESC
    LIMIT 1
""")
row = cur.fetchone()

if row:
    tid = row[0]
    cur.execute(f"""
        SELECT
            TRACE:span_id::STRING AS SPAN_ID,
            RECORD_ATTRIBUTES:"gen_ai.input.messages"::STRING AS INPUT_MSGS,
            RECORD_ATTRIBUTES:"gen_ai.output.messages"::STRING AS OUTPUT_MSGS,
            RECORD_ATTRIBUTES:"gen_ai.usage.input_tokens"::INT AS IN_TOK,
            RECORD_ATTRIBUTES:"gen_ai.usage.output_tokens"::INT AS OUT_TOK
        FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
        WHERE TRACE:trace_id::STRING = '{tid}'
        ORDER BY TIMESTAMP
    """)
    spans = cur.fetchall()
    print(f"Trace {tid} — {len(spans)} gateway call(s)\n")

    for i, span in enumerate(spans):
        msgs_in = json.loads(span[1]) if span[1] else []
        msgs_out = json.loads(span[2]) if span[2] else []
        print(f"═══ Gateway Call {i+1} ({span[3]} in / {span[4]} out tokens) ═══")

        for m in msgs_in:
            role = m.get('role', '?')
            parts = m.get('parts', [])
            for p in parts:
                if 'content' in p:
                    print(f"  [{role}] {p['content'][:200]}")
                elif 'name' in p:
                    print(f"  [{role}] → tool_call: {p['name']}({json.dumps(p.get('arguments',{}))[:150]})")
                elif 'response' in p:
                    resp_text = str(p['response'])[:200]
                    print(f"  [tool] ← {resp_text}")

        for m in msgs_out:
            parts = m.get('parts', [])
            for p in parts:
                if 'content' in p:
                    print(f"  [assistant] {p['content'][:300]}")
                elif 'name' in p:
                    print(f"  [assistant] → tool_call: {p['name']}({json.dumps(p.get('arguments',{}))[:150]})")
        print()
else:
    print("No multi-turn traces found in last 2 hours.")

## 9. Cost Management

The gateway is a natural unit for cost reporting: every inference request is attributed to the
Snowflake user who made it, with token counts and credit consumption recorded automatically.
This section covers how to monitor what the gateway is spending and how to limit it.

Snowflake offers two spending controls for gateway spend:

| Control | Scope | Enforcement | Use it when |
|---|---|---|---|
| **Shared resource budget** | A group of users identified by a tag, pooled | Notification, plus stored procedures you write | Several teams share the gateway and each needs its own limit |
| **Per-user quota** | Each user individually (limits are never pooled) | Built-in blocking, applied within minutes | You need usage to actually stop, not just alert |

Budget evaluation is periodic (up to 6.5 hours, or one hour with the low-latency option).
Quota blocks apply within minutes.

In [ ]:
# Gateway spend by user over the last 7 days
cur.execute("""
    SELECT
        u.NAME AS USER_NAME,
        f.key AS MODEL,
        COUNT(*) AS REQUEST_COUNT,
        SUM(f.value:"input_tokens"::INT) AS TOTAL_INPUT_TOKENS,
        SUM(f.value:"output_tokens"::INT) AS TOTAL_OUTPUT_TOKENS,
        SUM(g.CREDITS) AS TOTAL_CREDITS
    FROM SNOWFLAKE.ACCOUNT_USAGE.AI_GATEWAY_USAGE_HISTORY g
        JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u ON g.USER_ID = u.USER_ID
        ,LATERAL FLATTEN(input => g.OPERATION_DETAILS) f
    WHERE g.START_TIME > DATEADD('day', -7, CURRENT_TIMESTAMP())
    GROUP BY u.NAME, f.key
    ORDER BY TOTAL_CREDITS DESC
""")

rows = cur.fetchall()
print("AI Gateway Spend by User (last 7 days)\n")
print(f"{'USER':>20} | {'MODEL':>20} | {'REQUESTS':>8} | {'IN_TOKENS':>10} | {'OUT_TOKENS':>10} | {'CREDITS':>10}")
print("─" * 90)
for row in rows:
    print(f"{str(row[0]):>20} | {str(row[1]):>20} | {row[2]:>8} | {row[3]:>10} | {row[4]:>10} | {row[5]:>10.6f}")

### Shared resource budget

A shared resource budget lets you track gateway spend by team or cost center. The workflow is:

1. **Tag users** with a cost-center or team tag
2. **Create a budget** and add the tag so it tracks only those users
3. **Add the AI Gateway** as a shared resource on the budget
4. **Set a spending limit** — notifications fire when spend approaches it

When a tagged user makes a gateway request, the credits count toward that budget's spending limit.
You can configure notifications and custom actions (stored procedures) that fire at configurable thresholds.

In [ ]:
# --- Shared resource budget for gateway spend ---

# 1. Create a tag for cost-center attribution
cur.execute("""CREATE TAG IF NOT EXISTS CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.COST_CENTER
               COMMENT = 'Team or cost center for gateway budget attribution'""")
print("1. Tag created:", cur.fetchone()[0])

# 2. Tag the current user
cur.execute(f"""ALTER USER "{SNOWFLAKE_USER.upper()}" SET TAG CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.COST_CENTER = 'ENGINEERING'""")
print("2. User tagged:", cur.fetchone()[0])

# 3. Create the budget
cur.execute("""CREATE SNOWFLAKE.CORE.BUDGET IF NOT EXISTS CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.GATEWAY_BUDGET()""")
print("3. Budget created:", cur.fetchone()[0])

# 4. Associate the user tag with the budget
cur.execute("""CALL CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.GATEWAY_BUDGET!SET_USER_TAGS(
    [[(SELECT SYSTEM$REFERENCE('TAG', 'CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.COST_CENTER', 'SESSION', 'APPLYBUDGET')),
      'ENGINEERING']],
    'UNION')""")
print("4. User tag set:", cur.fetchone()[0])

# 5. Add AI Gateway as a shared resource
cur.execute("""CALL CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.GATEWAY_BUDGET!ADD_SHARED_RESOURCE('AI GATEWAY')""")
print("5. Shared resource added:", cur.fetchone()[0])

# 6. Set a 500-credit monthly spending limit
cur.execute("""CALL CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.GATEWAY_BUDGET!SET_SPENDING_LIMIT(500)""")
print("6. Spending limit set:", cur.fetchone()[0])

# Verify the budget scope
cur.execute("""CALL CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.GATEWAY_BUDGET!GET_BUDGET_SCOPE()""")
scope = cur.fetchone()[0]
print("\nBudget scope:")
print(json.dumps(json.loads(scope), indent=2))

### Per-user quota

A per-user quota enforces hard spending limits at the individual user level. Unlike budgets,
quotas can **block** users automatically when they reach their limit — no custom stored procedure
needed. Blocks are released when the cycle resets (monthly, weekly, or daily).

Per-user quotas support monthly, weekly, and daily limits evaluated independently. A user is
blocked as soon as they hit any one of them. Block enforcement is evaluated within minutes of
a spend event.

In [ ]:
# --- Per-user quota for gateway spend ---

# 1. Create the quota object
cur.execute("""CREATE SNOWFLAKE.CORE.QUOTA IF NOT EXISTS CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.GATEWAY_QUOTA()""")
print("1. Quota created:", cur.fetchone()[0])

# 2. Add AI Gateway as the monitored domain
cur.execute("""CALL CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.GATEWAY_QUOTA!ADD_SHARED_RESOURCE('AI GATEWAY')""")
print("2. Shared resource added:", cur.fetchone()[0])

# 3. Set per-user spending limits at each cycle
cur.execute("""CALL CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.GATEWAY_QUOTA!SET_PER_USER_LIMIT(100)""")           # monthly
print("3a. Monthly limit:", cur.fetchone()[0])

cur.execute("""CALL CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.GATEWAY_QUOTA!SET_PER_USER_LIMIT(30, 'WEEKLY')""")  # weekly
print("3b. Weekly limit:", cur.fetchone()[0])

cur.execute("""CALL CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.GATEWAY_QUOTA!SET_PER_USER_LIMIT(10, 'DAILY')""")   # daily
print("3c. Daily limit:", cur.fetchone()[0])

# 4. Enable block enforcement — users are blocked when they hit any limit
cur.execute("""CALL CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.GATEWAY_QUOTA!SET_BLOCK_ENFORCEMENT_ENABLED(TRUE, FALSE)""")
print("4. Block enforcement:", cur.fetchone()[0])

# Verify the quota config
cur.execute("""CALL CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.GATEWAY_QUOTA!GET_CONFIG()""")
row = cur.fetchone()
cols = [desc[0] for desc in cur.description]
print("\nQuota config:")
for col, val in zip(cols, row):
    print(f"  {col}: {val}")

### Watch enforcement

`setup.sql` section 9b creates `GATEWAY_RETAIL_QUOTA` (1 credit per day, block enforcement on) for a single service user, `GATEWAY_RETAIL_COST_DEMO`. Run `python lab/generate_traffic.py --burst` to spend as that user until the gateway denies it with HTTP 403. Enforcement lands within minutes of the spend, so the user ends up slightly over the limit.

The cell below shows per-user spend against the daily limit, who is blocked now, and the block history.

In [ ]:
import datetime as dt

QUOTA = "CORTEX_GATEWAY_RETAIL_LAB.PUBLIC.GATEWAY_RETAIL_QUOTA"
today = dt.date.today().isoformat()


def show(sql, title):
    cur.execute(sql)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    print(f"\n{title} ({len(rows)} rows)")
    for r in rows:
        print("  " + ", ".join(f"{c}={v}" for c, v in zip(cols, r)))


# Finalized spend for users in the quota today (credits, by hour)
show(f"CALL {QUOTA}!GET_SPENDING_DETAILS_BY_USERS('{today}', '{today}')", "Spend today")

# Users the quota is blocking right now, and until when
show(f"CALL {QUOTA}!GET_ACTIVE_BLOCKS_V2()", "Active blocks")

# Every block and unblock across all quotas (ACCOUNT_USAGE, can lag)
show("""SELECT ACTION_AT, QUOTA_NAME, USER_NAME, CYCLE, ACTION, PER_USER_LIMIT, CREDITS, BLOCKED_UNTIL
        FROM SNOWFLAKE.ACCOUNT_USAGE.QUOTA_ACCESS_BLOCK_HISTORY
        ORDER BY ACTION_AT DESC LIMIT 10""", "Block history")

## Summary

This example demonstrated:

| Component | Role |
|---|---|
| **Cortex AI Gateway** | Centralized LLM routing — one endpoint, configurable routes, built-in observability |
| **LangChain + LangGraph** | Agent framework with ReAct reasoning over MCP tools |
| **MCP (Model Context Protocol)** | Standardized tool interface connecting the agent to Snowflake data services |
| **Cortex Analyst** | Natural language → SQL against two semantic views (customer + churn, executive KPIs) |
| **Model Registry** | XGBoost churn model `CHURN_PREDICTION_MODEL/V1`, batch-scored in the warehouse into `CHURN_PREDICTIONS` and watched by `CHURN_MODEL_MONITOR` |
| **AGENT_TRACE_TABLE** | OpenTelemetry traces for every gateway inference call |
| **AI_GATEWAY_USAGE_HISTORY** | Token usage and credit consumption per model/route |
| **Budgets & Per-user Quotas** | Spending controls — team-level budgets with notifications, or per-user hard blocks |

### Key Takeaways

1. **The gateway is OpenAI-compatible** — any SDK or framework that speaks the OpenAI chat completions API works by pointing `base_url` at `/api/v2/cortex/v1`.
2. **Model allowlists control access** — the `models` spec restricts which models can be called; use `'*'` for unrestricted or `'claude-*'` to lock down to specific families.
3. **MCP provides a clean tool boundary** — the agent doesn't need Snowflake credentials or SQL knowledge; MCP handles that.
4. **Observability is automatic** — every inference call is traced in `AGENT_TRACE_TABLE('SNOWFLAKE')` with token counts, model metadata, and optional payload capture.
5. **Cost controls are built in** — shared resource budgets track team-level spend with notifications; per-user quotas enforce hard blocks within minutes when a user hits their credit limit.